# 예제 I · Standard Conv2d ResNet-18 (수강생용)

BasicBlock의 첫 번째 공간 연산을 일반 3×3 Conv2d로 구성합니다. 코드를 완성한 뒤 첫 Stage 전환의 채널과 기준 모델의 계산량을 확인합니다.

## 1. 실행 환경과 CIFAR-10

첫 코드 셀에서 실행 장치와 데이터 경로를 확인합니다. 최초 실행에는 CIFAR-10을 내려받는 시간이 포함됩니다.

- CIFAR-10은 **32×32 RGB 영상, 10개 클래스**로 구성됩니다. 입력을 Tensor로 바꾼 뒤 RGB 채널별 평균과 표준편차로 정규화합니다.
- 학습과 평가는 CIFAR-10의 서로 다른 분할에서 진행합니다. 난수 시드를 고정해 테스트 영상 **512장**을 같은 방식으로 선택합니다.
- 학습 이미지 **10,000장**, 배치 크기 **64**, 테스트 **512장**을 사용합니다. `TRAIN_EPOCHS`의 기본값은 **3**이며 **5**도 선택할 수 있습니다.

**코드 셀 1 · 실행 환경과 학습 설정**

실행 장치와 경로를 설정하고, 고정 학습 이미지 10,000장과 테스트 512장을 선택할 조건을 정합니다.

In [ ]:
# @title 학습 실행 설정
# 실습 설정을 한곳에서 관리한다. 데이터·가중치 캐시를 재사용하고 CUDA가 있으면 GPU로 실행한다.
import copy
import os
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# 동일한 시드와 배치 수를 사용해 반복 실행 조건을 맞춘다.
SEED = 50
BATCH_SIZE = 64
TRAIN_SAMPLES = 10_000
TRAIN_EPOCHS = 3 # @param {type:"slider", min:3, max:5, step:2}
TEST_SAMPLES = 512
DATA_ROOT = Path(os.environ.get("AX_IDEC_DATA_ROOT", "./data"))
ASSET_ROOT = Path(os.environ.get("AX_IDEC_CACHE", "./idec_student_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("실행 장치:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU")

**코드 셀 2 · CIFAR-10과 DataLoader 준비**

CIFAR-10을 RGB 기준으로 정규화하고 고정된 시드로 학습과 테스트에 사용할 데이터를 선택합니다.

이미 내려받은 CIFAR-10은 런타임의 캐시에서 읽습니다. 첫 실행에서는 공유 Google Drive에서 받고, 실패하면 CIFAR-10의 공식 주소를 사용합니다.

학습용 1만 장을 고정하고 epoch마다 배치 순서를 섞습니다. 예제 I·II에서 같은 이미지와 순서를 사용합니다.

In [ ]:
# CIFAR-10 학습·테스트 분할에 같은 RGB 정규화를 적용한다. 테스트는 학습에 쓰이지 않는다.
CIFAR_MEAN = (0.49139968, 0.48215841, 0.44653091)
CIFAR_STD = (0.24703223, 0.24348513, 0.26158784)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
from http.client import HTTPException
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from urllib.request import urlopen

from torchvision.datasets.utils import check_integrity


class DriveCIFAR10(datasets.CIFAR10):
    # 압축 해제된 데이터나 검증된 압축 파일을 먼저 재사용한다.
    def download(self):
        if self._check_integrity():
            return

        root = Path(self.root)
        archive = root / self.filename
        if not check_integrity(archive, self.tgz_md5):
            drive_url = (
                "https://drive.usercontent.google.com/download"
                "?id=1Sx382Bt_XcAUj9_igoCgWoNQ6c5ooSa6&export=download&confirm=t"
            )
            try:
                root.mkdir(parents=True, exist_ok=True)
                with TemporaryDirectory(dir=root) as temp_dir:
                    staged = Path(temp_dir) / self.filename
                    print("Google Drive에서 CIFAR-10을 내려받습니다.")
                    with urlopen(drive_url, timeout=30) as response:
                        if response.headers.get_content_type() == "text/html":
                            raise ValueError("Drive가 파일 대신 안내 페이지를 반환했습니다.")
                        with staged.open("wb") as output:
                            copyfileobj(response, output, length=1024 * 1024)
                    if not check_integrity(staged, self.tgz_md5):
                        raise ValueError("CIFAR-10 원본 체크섬과 일치하지 않습니다.")
                    staged.replace(archive)
            except (OSError, HTTPException, ValueError):
                print("Drive에서 CIFAR-10을 받지 못해 공식 다운로드 주소로 다시 시도합니다.")

        # torchvision이 원본 체크섬을 확인하고 필요한 경우 압축을 해제한다.
        super().download()


train_data = DriveCIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_data = DriveCIFAR10(DATA_ROOT, train=False, download=True, transform=transform)

# 고정된 학습 샘플을 선택하고, 학습 순서만 매 epoch 섞는다.
def make_loader(dataset, samples, seed, *, shuffle=False):
    indices = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(seed))
    subset = Subset(dataset, indices[:samples].tolist())
    return DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )

train_loader = make_loader(train_data, TRAIN_SAMPLES, SEED, shuffle=True)
test_loader = make_loader(test_data, TEST_SAMPLES, SEED + 1)
print(f"학습 이미지 {len(train_loader.dataset):,}장 / epoch당 {len(train_loader)}배치 / 테스트 {len(test_loader.dataset)}장")


## 2. TODO · BasicBlock의 첫 번째 Conv2d 정의

- **수정할 부분** · 바로 아래 **코드 셀 3**의 `BasicBlock.__init__()`에서 `self.conv1`을 입력 채널 `inplanes`, 출력 채널 `planes`를 갖는 3×3 Conv2d로 정의합니다. 블록에 전달된 `stride`를 사용합니다.
- **실행 후 확인** · 셀 3–4에서 첫 Stage 전환의 출력 크기를 확인하고, 셀 5–6에서 MACs와 파라미터 수를 확인합니다.

**코드 셀 3 · ResNet-18의 BasicBlock 구성**

BasicBlock의 첫 번째 3×3 Conv2d를 완성하고 Stage마다 BasicBlock 두 개를 배치한 ResNet-18을 정의합니다.

In [3]:
# 예제 I의 기준 모델이다. 네 Residual stage에 BasicBlock을 두 개씩 사용한다.
def build_resnet18():
    __all__ = ['ResNet', 'resnet18']

    def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
        """3x3 convolution with padding"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                         padding=dilation, groups=groups, bias=False, dilation=dilation)

    def conv1x1(in_planes, out_planes, stride=1):
        """1x1 convolution"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

    # 주 경로에서 특징을 계산하고 Residual 입력을 더한다. 크기가 바뀌는 블록에서는 Residual도 변환한다.
    class BasicBlock(nn.Module):
        expansion = 1

        def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                     base_width=64, dilation=1, norm_layer=None):
            super(BasicBlock, self).__init__()
            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            if groups != 1 or base_width != 64:
                raise ValueError(
                    'BasicBlock only supports groups=1 and base_width=64')
            if dilation > 1:
                raise NotImplementedError(
                    "Dilation > 1 not supported in BasicBlock")
            # Both self.conv1 and self.downsample layers downsample the input when stride != 1
            # ========================================================
            # TODO: BasicBlock의 첫 번째 3×3 Conv2d를 정의한다.

            self.conv1 = conv3x3(inplanes, planes, stride)  # TODO

            # ========================================================
            if self.conv1 is None:
                raise ValueError('코드 셀 3의 BasicBlock TODO를 먼저 채우세요.')
            self.bn1 = norm_layer(planes)
            self.relu = nn.ReLU(inplace=True)
            self.conv2 = conv3x3(planes, planes)
            self.bn2 = norm_layer(planes)
            self.downsample = downsample
            self.stride = stride

        def forward(self, x):
            # 원래 입력을 Residual 경로에 보관한다. 주 경로의 두 연산이 끝난 뒤 더한다.
            identity = x

            out = self.conv1(x)
            out = self.bn1(out)
            out = self.relu(out)

            out = self.conv2(out)
            out = self.bn2(out)

            if self.downsample is not None:
                identity = self.downsample(x)

            out += identity
            out = self.relu(out)

            return out

    # 입력 stem과 Residual stage를 연결하고, Global Average Pooling 뒤에서 클래스별
    # Logit을 계산한다.
    class ResNet(nn.Module):

        def __init__(self, block, layers, num_classes=1000, zero_init_residual=False,
                     groups=1, width_per_group=64, replace_stride_with_dilation=None,
                     norm_layer=None):
            super(ResNet, self).__init__()
            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            self._norm_layer = norm_layer

            self.inplanes = 64
            self.dilation = 1
            if replace_stride_with_dilation is None:
                # each element in the tuple indicates if we should replace
                # the 2x2 stride with a dilated convolution instead
                replace_stride_with_dilation = [False, False, False]
            if len(replace_stride_with_dilation) != 3:
                raise ValueError("replace_stride_with_dilation should be None "
                                 "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
            self.groups = groups
            self.base_width = width_per_group
            self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3,
                                   bias=False)

            self.bn1 = norm_layer(self.inplanes)
            self.relu = nn.ReLU(inplace=True)
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            # 네 stage는 64·128·256·512채널을 사용한다. 첫 블록의 stride로 해상도를 조정한다.
            self.layer1 = self._make_layer(block, 64, layers[0])
            self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                           dilate=replace_stride_with_dilation[0])
            self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                           dilate=replace_stride_with_dilation[1])
            self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                           dilate=replace_stride_with_dilation[2])
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
            self.fc = nn.Linear(512 * block.expansion, num_classes)

            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        m.weight, mode='fan_out', nonlinearity='relu')
                elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)

            # Zero-initialize the last BN in each residual branch,
            # so that the residual branch starts with zeros, and each residual block behaves like an identity.
            # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
            if zero_init_residual:
                # Convolution과 정규화 계층의 학습 초기값을 설정한다.
                for m in self.modules():
                    if isinstance(m, Bottleneck):
                        nn.init.constant_(m.bn3.weight, 0)
                    elif isinstance(m, BasicBlock):
                        nn.init.constant_(m.bn2.weight, 0)

        # 첫 블록에서 필요한 Residual 변환을 적용하고, 뒤의 블록은 같은 출력 크기로 이어 붙인다.
        def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
            norm_layer = self._norm_layer
            downsample = None
            previous_dilation = self.dilation
            if dilate:
                self.dilation *= stride
                stride = 1
            if stride != 1 or self.inplanes != planes * block.expansion:
                downsample = nn.Sequential(
                    conv1x1(self.inplanes, planes * block.expansion, stride),
                    norm_layer(planes * block.expansion),
                )

            layers = []
            layers.append(block(self.inplanes, planes, stride, downsample, self.groups,
                                self.base_width, previous_dilation, norm_layer))
            self.inplanes = planes * block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, planes, groups=self.groups,
                                    base_width=self.base_width, dilation=self.dilation,
                                    norm_layer=norm_layer))

            return nn.Sequential(*layers)

        # 입력에서 stem, 네 stage, Pooling, 최종 Linear Layer로 이어지는 순전파다.
        def _forward_impl(self, x):
            # See note [TorchScript super()]
            x = self.conv1(x)
            x = self.bn1(x)
            x = self.relu(x)
            x = self.maxpool(x)

            x = self.layer1(x)
            x = self.layer2(x)
            x = self.layer3(x)
            x = self.layer4(x)

            x = self.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.fc(x)

            return x

        def forward(self, x):
            return self._forward_impl(x)

    # CIFAR-10에 맞게 첫 Conv2d를 3×3, stride 1로 설정하고 출력 클래스 수를 10으로 둔다.
    def resnet_18_cifar():
        r"""
            Modify the resnet 18 network in order to run on cifar-10 dataset

            To enhance the accuracy, the (kernel_size, stride, padding) of conv1 is modified to (3, 1, 1)
            referenced by <https://github.com/akamaster/pytorch_resnet_cifar10>
        """
        model = ResNet(block=BasicBlock, layers=[2, 2, 2, 2], num_classes=10)
        model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1,
                                bias=False)

        return model

    return resnet_18_cifar()

**코드 셀 4 · BasicBlock의 입출력 확인**

첫 Stage 전환에 예제 Tensor를 넣어 Conv2d의 입출력 채널 수와 BasicBlock 출력 크기를 확인합니다.

In [4]:
# 첫 Stage 전환에서 채널과 공간 크기가 어떻게 바뀌는지 확인한다.
model = build_resnet18().to(DEVICE).eval()
first_transition = model.layer2[0]
block_input = torch.zeros(1, 64, 16, 16, device=DEVICE)
with torch.inference_mode():
    block_output = first_transition(block_input)
print(f"Standard Conv2d 채널: {first_transition.conv1.in_channels} → {first_transition.conv1.out_channels}")
print(f"BasicBlock 입력 → 출력: {tuple(block_input.shape)} → {tuple(block_output.shape)}")

ValueError: 코드 셀 3의 BasicBlock TODO를 먼저 채우세요.

## 3. 기준 모델의 MACs와 파라미터 수

코드 셀 5–6을 실행합니다. 3×32×32 이미지 한 장당 MACs와 전체 파라미터 수를 예제 II와 비교할 기준으로 기록합니다.

**코드 셀 5 · 학습·평가·연산량 측정 함수 정의**

모델의 MACs와 파라미터 수, 분류 정확도, epoch별 학습 loss를 계산하는 함수를 정의합니다.

In [ ]:
# 평가 모드에서 각 영상의 최댓값 클래스를 정답과 비교해 전체 정확도를 집계한다.
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (model(images).argmax(1) == labels).sum().item()
        total += labels.numel()
    return 100 * correct / total

# Conv2d와 Linear에 Hook을 붙여 이미지 한 장의 Dense MACs를 구하고 전체 파라미터 수를 센다.
def macs_and_parameters(model, input_shape=(3, 32, 32)):
    # Conv2d와 Linear의 multiply-accumulate만 집계.
    operations = 0
    def count(module, args, output):
        nonlocal operations
        if isinstance(module, nn.Conv2d):
            kernel = module.kernel_size[0] * module.kernel_size[1]
            operations += output.numel() * (module.in_channels // module.groups) * kernel
        else:
            operations += output.numel() * module.in_features
    # 실행 중 필요한 계층에만 Hook을 연결하고 블록이 끝나면 제거한다.
    hooks = [m.register_forward_hook(count) for m in model.modules()
             if isinstance(m, (nn.Conv2d, nn.Linear))]
    previous_mode = model.training
    try:
        model.eval()
        with torch.inference_mode():
            model(torch.zeros(1, *input_shape, device=DEVICE))
    finally:
        for hook in hooks:
            hook.remove()
        model.train(previous_mode)
    return operations, sum(p.numel() for p in model.parameters())

# 한 번 만든 Optimizer를 모든 epoch에서 사용한다.
def train_epochs(model, loader, test_loader, epochs):
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    history = []
    train_seconds = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, samples = 0.0, 0
        start = time.perf_counter()

        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(images), labels)
            loss.backward()
            optimizer.step()
            # 마지막 배치가 작아도 이미지마다 같은 비중으로 평균을 낸다.
            loss_sum += loss.item() * labels.size(0)
            samples += labels.size(0)

        train_seconds += time.perf_counter() - start
        average_loss = loss_sum / samples
        accuracy = evaluate(model, test_loader)
        history.append((average_loss, accuracy))
        print(f"Epoch {epoch}/{epochs} · 평균 학습 loss {average_loss:.3f} · 테스트 정확도 {accuracy:.2f}%")

    return history, train_seconds


**코드 셀 6 · ResNet-18의 MACs와 파라미터 수**

CIFAR-10 크기인 3×32×32의 입력 한 장을 기준으로 ResNet-18의 MACs와 파라미터 수를 출력합니다.

In [ ]:
# 학습과 별개로 모델 구조의 연산량과 파라미터 수를 계산한다.
macs, parameters = macs_and_parameters(model)
print(f"MACs/image: {macs / 1e6:.2f} M | Parameters: {parameters / 1e6:.2f} M")

## 4. 학습과 평가

`TRAIN_EPOCHS=3`이면 같은 학습 이미지 10,000장을 세 번 순회합니다. 매 epoch의 평균 Cross-Entropy loss와 고정된 CIFAR-10 테스트 이미지 512장의 정확도를 확인합니다. 학습이 끝나면 가중치를 `idec_resnet18_short_run.pt`로 저장합니다.

5 epoch을 실행하려면 **셀 1에서 `TRAIN_EPOCHS`를 5로 선택**한 뒤 셀 1과 셀 7을 다시 실행합니다. 학습은 같은 초기 조건에서 시작합니다. 예제 I·II를 비교할 때는 두 노트북의 epoch 수를 맞춥니다.

**코드 셀 7 · 기준 ResNet-18의 epoch별 학습과 평가**

epoch마다 학습 loss와 테스트 정확도를 출력하고 마지막에 총 학습 시간을 표시합니다.

In [ ]:
# epoch 수만 바꿔 다시 실행해도 모델과 학습 순서는 같은 초기 조건에서 시작한다.
torch.manual_seed(SEED)
train_loader.generator.manual_seed(SEED)
model = build_resnet18().to(DEVICE)
epoch_history, seconds = train_epochs(model, train_loader, test_loader, TRAIN_EPOCHS)
accuracy = epoch_history[-1][1]
print(f"총 학습 시간 {seconds:.1f} s")
checkpoint = ASSET_ROOT / "idec_resnet18_short_run.pt"
torch.save(model.state_dict(), checkpoint)
print("가중치 저장:", checkpoint)


## 추가로 바꿔볼 설정 · 선택 실습

- `TRAIN_EPOCHS`를 3에서 5로 바꾸고 셀 1과 셀 7을 다시 실행합니다. epoch별 loss, 테스트 정확도, 총 학습 시간을 비교합니다.
- 다른 조건을 고정한 채 셀 5에서 학습률 `lr`을 바꿔 해당 셀과 셀 7을 다시 실행합니다. loss와 정확도가 어떻게 달라지는지 확인합니다.